In [ ]:
from preprocessing import dataset_prepocessing 

from preprocessing import detect_categories 

from tree import DecisionTree 

from sklearn.model_selection import train_test_split

from itertools import product 

import pandas as pd

def training_split(dataset) : 

    train_data , test_data = train_test_split(dataset , test_size = 0.2 , random_state = 42 , stratify = dataset['satisfaction'])     
    return train_data , test_data

def tuning_split(train_data) :
    
    tuning_train , tuning_validation = train_test_split(train_data , test_size = 0.2 , random_state = 42 , stratify = train_data['satisfaction'])
    return tuning_train , tuning_validation 

def hyperparameter_tuning(train_data , feats_dict):

    tuning_train , tuning_validation = tuning_split(train_data)
    hyper_params_df = {"max_depth" : [] , "min_samples_split" : [] ,"soft_max_leaf_nodes" : [] ,"soft_min_samples_leaf" : [] , "criterion" : [] , "F1-Score" : []}
    max_depth_list = [3,5,7,9,12,None]
    min_samples_split_list = [2,5,10,20]
    soft_max_leaf_nodes_list = [None,20,40]
    soft_min_samples_leaf_list = [1,5,10]
    criterion_list = ['gini','gain']

    for md,mss,mln,msl,crit in product(max_depth_list,min_samples_list,soft_max_leaf_nodes_list,soft_min_samples_leaf_list,criterion_list):
        hyper_params = {"max_depth" : md , "min_samples_split" : mss , "soft_max_leaf_nodes" : mln , "soft_min_samples_leaf" : msl , "criterion" : crit}
        decision_tree = DecisionTree()
        decision_tree.training(tuning_train , hyper_params , feats_dict)
        F1_Score = decision_tree.F1_evaluation(tuning_validation)
        hyper_params_df["max_depth"].append(md)
        hyper_params_df["min_samples_split"].append(mss)
        hyper_params_df["soft_max_leaf_nodes"].append(mln)
        hyper_params_df["soft_min_samples_leaf"].append(msl)
        hyper_params_df["criterion"].append(crit)
        hyper_params_df["F1-Score"].append(F1_Score)
    
        # pruning 
    hyper_params_df = pd.DataFrame(hyper_params_df)
    optimal_row = hyper_params_df["F1-Score"].idxmax()
    
    optimal_hyper_params = {}
    for hyper_param in hyper_params.keys():
        optimal_hyper_params[hyper_param] = hyper_params_df.loc[optimal_row , hyper_param ]

    hyper_params_df.to_csv("../data/hyper_params.csv" , index = False)

    return optimal_hyper_params


def training(train_data , test_data , optimal_hyper_params , feats_dict):

    decision_tree = DecisionTree()
    decision_tree.training(train_data , optimal_hyper_params , feats_dict)
    F1_Score = decision_tree.F1_evaluation(test_data)
    # pruning 

    return decision_tree 
